# 연별 노선×정류소 이용량(AnnualRoutebyStopTripVolume) → DuckDB 적재

공공데이터포털 `RoutebyStopTripVolume/getAnnualRoutebyStopTripVolume`.

**프로젝트 핵심 팩트 테이블**: 연간 × 시간대 × 이용자유형별 승하차 인원.

- `opr_yr` = **2025**
- `ctpv_cd` = 11 (서울), `sgg_cd` = 서울 25개 구 → **시군구별 1회 호출**
- `rte_id` 선택적 → 빼고 호출하면 시군구 전체 노선이 한 번에 반환
- **호출 수**: 서울 25구 × 1년 = **25 API 호출** (빠름)
- **재개 지원**: (opr_yr, sgg_cd) 단위
- PK: (opr_yr, rte_id, sttn_seq, sttn_id, users_type_cd, tzon)

In [1]:
# ============================================================
# 셀 1 - 환경 설정 + DB 헬퍼 정의
# ============================================================
import os
import time
import requests
import duckdb
import pandas as pd
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv()
SERVICE_KEY = unquote(os.environ["MY_SERVICE_KEY"])

DB_PATH = "data/seoul.duckdb"
Path("data").mkdir(exist_ok=True)


@contextmanager
def db_open(read_only: bool = False):
    """짧게 연결하고 무조건 닫는 컨텍스트 매니저 (예외 시에도 안전)."""
    con = duckdb.connect(DB_PATH, read_only=read_only)
    try:
        yield con
    finally:
        con.close()


print(f"SERVICE_KEY 로드됨 (길이 {len(SERVICE_KEY)})")
print(f"DB: {DB_PATH} (헬퍼 준비됨)")

SERVICE_KEY 로드됨 (길이 88)
DB: data/seoul.duckdb (헬퍼 준비됨)


In [2]:
# ============================================================
# 셀 2 - Pydantic 모델
# ============================================================
from typing import Generic, TypeVar, Optional
from pydantic import BaseModel, field_validator

T = TypeVar("T", bound=BaseModel)


class Header(BaseModel):
    resultCode: str
    resultMsg: str


class Items(BaseModel, Generic[T]):
    item: list[T] = []

    @field_validator("item", mode="before")
    @classmethod
    def _coerce_to_list(cls, v):
        if v is None or v == "":
            return []
        if isinstance(v, dict):
            return [v]
        return v


class Body(BaseModel, Generic[T]):
    items: Items[T] = Items[T]()
    dataType: str | None = None
    pageNo: int
    numOfRows: int
    totalCount: int


class Response_(BaseModel, Generic[T]):
    header: Header
    body: Body[T]


class PublicDataResponse(BaseModel, Generic[T]):
    Response: Response_[T]


class AnnualTripVolumeItem(BaseModel):
    """연별 노선×정류소 이용량 한 건 (시간대 × 이용자유형별 승하차).

    PK: (opr_yr, rte_id, sttn_seq, sttn_id, users_type_cd, tzon)

    특이사항:
      - emd_cd / sttn_id 가 '~'로 올 수 있음 → 그대로 문자열 저장
      - nm 필드는 region/user_type JOIN으로 복원 → 저장 안 함 (정규화)
    """
    # 필수 (PK 구성)
    opr_yr: str
    ctpv_cd: str
    rte_id: str
    sttn_seq: int
    sttn_id: str
    users_type_cd: str
    tzon: str

    # 코드 (region JOIN용)
    sgg_cd: Optional[str] = None
    emd_cd: Optional[str] = None

    # 지표
    ride_nope: int = 0    # 승차 인원
    goff_nope: int = 0    # 하차 인원

    # 응답에만 있고 저장 안 함
    ctpv_nm: Optional[str] = None
    sgg_nm: Optional[str] = None
    emd_nm: Optional[str] = None
    users_type_nm: Optional[str] = None


print("모델 로드 완료")

모델 로드 완료


In [3]:
# ============================================================
# 셀 3 - fetch 함수 (NO_DATA_FOUND 에러 처리 포함)
# ============================================================
ANNUAL_TRIP_URL = "https://apis.data.go.kr/1613000/RoutebyStopTripVolume/getAnnualRoutebyStopTripVolume"


def fetch_all_pages(
    url: str,
    item_model: type[T],
    extra_params: dict | None = None,
    num_rows: int = 1000,
) -> list[T]:
    """공공데이터 표준 응답 API 전체 페이지 수집. NO_DATA_FOUND는 빈 결과로 취급."""
    params_base = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_rows,
        "dataType": "JSON",
        **(extra_params or {}),
    }
    all_items: list[T] = []
    page = 1
    while True:
        res = requests.get(url, params={**params_base, "pageNo": page}, timeout=60)
        res.raise_for_status()
        payload = res.json()

        if "Error" in payload:
            err = payload["Error"]
            code = err.get("code")
            msg = err.get("message", "")
            if msg == "NO_DATA_FOUND" or code == "50":
                return all_items
            raise RuntimeError(f"API error: {code} {msg}")

        parsed = PublicDataResponse[item_model].model_validate(payload)
        header = parsed.Response.header
        if header.resultCode not in ("00", "200"):
            raise RuntimeError(f"API error: {header.resultCode} {header.resultMsg}")

        body = parsed.Response.body
        items = body.items.item
        all_items.extend(items)

        if len(all_items) >= body.totalCount or not items:
            break
        page += 1
        time.sleep(0.05)

    return all_items


print("fetch 함수 준비 완료")

fetch 함수 준비 완료


In [4]:
# ============================================================
# 셀 4 - 파라미터 준비: 서울 25구 + 테이블 + 재개 목록
# ============================================================
# API 파라미터: opr_yr + ctpv_cd + sgg_cd (rte_id는 선택이라 생략)
# → 시군구 하나 호출이 그 구의 모든 노선/정류소 커버

OPR_YR = "2025"
print(f"대상 연도 opr_yr = {OPR_YR}")

with db_open() as con:
    con.execute("""
    CREATE TABLE IF NOT EXISTS annual_trip_volume (
        opr_yr         TEXT,
        ctpv_cd        TEXT,
        sgg_cd         TEXT,
        emd_cd         TEXT,
        rte_id         TEXT,
        sttn_seq       INTEGER,
        sttn_id        TEXT,
        users_type_cd  TEXT,
        tzon           TEXT,
        ride_nope      INTEGER,
        goff_nope      INTEGER,
        PRIMARY KEY (opr_yr, rte_id, sttn_seq, sttn_id, users_type_cd, tzon)
    )
    """)

with db_open(read_only=True) as con:
    sgg_list_df = con.execute("""
        SELECT sido_cd,
               sgg_cd,
               sido_cd || sgg_cd AS ctpv_sgg_5,
               locatadd_nm       AS sgg_nm
        FROM region
        WHERE sido_cd = '11'
          AND sgg_cd <> '000'
          AND umd_cd = '000'
          AND ri_cd = '00'
        ORDER BY sgg_cd
    """).df()

    done_df = con.execute("""
        SELECT DISTINCT sgg_cd
        FROM annual_trip_volume
        WHERE opr_yr = ?
    """, [OPR_YR]).df()

done_set = set(done_df["sgg_cd"]) if len(done_df) > 0 else set()
remaining = sgg_list_df[~sgg_list_df["ctpv_sgg_5"].isin(done_set)].reset_index(drop=True)

print(f"\n서울 시군구: {len(sgg_list_df)}개")
print(f"이미 완료: {len(sgg_list_df) - len(remaining)}개")
print(f"남은 작업: {len(remaining)}개")

if len(remaining) > 0:
    print(f"\n처리 예정:")
    print(remaining.head(10))

대상 연도 opr_yr = 2025

서울 시군구: 25개
이미 완료: 0개
남은 작업: 25개

처리 예정:
  sido_cd sgg_cd ctpv_sgg_5      sgg_nm
0      11    110      11110   서울특별시 종로구
1      11    140      11140    서울특별시 중구
2      11    170      11170   서울특별시 용산구
3      11    200      11200   서울특별시 성동구
4      11    215      11215   서울특별시 광진구
5      11    230      11230  서울특별시 동대문구
6      11    260      11260   서울특별시 중랑구
7      11    290      11290   서울특별시 성북구
8      11    305      11305   서울특별시 강북구
9      11    320      11320   서울특별시 도봉구


In [5]:
# ============================================================
# 셀 5 - 서울 시군구 순회, 시군구 단위 즉시 DB 저장
# ============================================================
# rte_id 파라미터 없이 sgg_cd로만 호출 → 시군구 전체 노선 한 번에
# 각 시군구 fetch 직후 바로 DB INSERT → 중간 중단 시에도 보존

failed: list[tuple[str, str, str]] = []    # (opr_yr, sgg_5, error)
started_at = datetime.now()
processed_total = 0

for i, row in enumerate(remaining.itertuples(index=False), start=1):
    sgg_5  = row.ctpv_sgg_5
    sgg_nm = row.sgg_nm

    try:
        items = fetch_all_pages(
            ANNUAL_TRIP_URL,
            AnnualTripVolumeItem,
            extra_params={
                "opr_yr":  OPR_YR,
                "ctpv_cd": "11",
                "sgg_cd":  sgg_5,
            },
        )

        if items:
            df_rows = pd.DataFrame([
                {
                    "opr_yr":        it.opr_yr,
                    "ctpv_cd":       it.ctpv_cd,
                    "sgg_cd":        it.sgg_cd,
                    "emd_cd":        it.emd_cd,
                    "rte_id":        it.rte_id,
                    "sttn_seq":      it.sttn_seq,
                    "sttn_id":       it.sttn_id,
                    "users_type_cd": it.users_type_cd,
                    "tzon":          it.tzon,
                    "ride_nope":     it.ride_nope,
                    "goff_nope":     it.goff_nope,
                }
                for it in items
            ])
            df_rows = df_rows.drop_duplicates(
                subset=["opr_yr","rte_id","sttn_seq","sttn_id","users_type_cd","tzon"]
            )
        else:
            df_rows = pd.DataFrame()

        if len(df_rows) > 0:
            with db_open() as con:
                con.register("df_view", df_rows)
                con.execute("""
                    INSERT INTO annual_trip_volume
                    SELECT opr_yr, ctpv_cd, sgg_cd, emd_cd,
                           rte_id, sttn_seq, sttn_id,
                           users_type_cd, tzon,
                           ride_nope, goff_nope
                    FROM df_view
                    ON CONFLICT (opr_yr, rte_id, sttn_seq, sttn_id, users_type_cd, tzon)
                      DO NOTHING
                """)
                con.unregister("df_view")

        processed_total += len(df_rows)
        status = f"+{len(df_rows):>7,}건"
    except Exception as e:
        failed.append((OPR_YR, sgg_5, str(e)[:100]))
        status = f"FAILED ({type(e).__name__})"

    elapsed = datetime.now() - started_at
    pct = i / len(remaining) * 100
    eta_sec = elapsed.total_seconds() / i * (len(remaining) - i)
    print(f"[{i:2d}/{len(remaining)}] {pct:5.1f}% | {sgg_5} {sgg_nm:18s} {status:18s} "
          f"| 누적 {processed_total:>10,} | 경과 {str(elapsed).split('.')[0]} | 잔여 ~{int(eta_sec//60)}분")

    time.sleep(0.15)

print(f"\n✅ 수집 완료")
print(f"   이번 실행 적재: {processed_total:,}건")
print(f"   실패 시군구: {len(failed)}개")
print(f"   총 소요시간: {datetime.now() - started_at}")

with db_open(read_only=True) as con:
    grand_total = con.execute("SELECT COUNT(*) FROM annual_trip_volume").fetchone()[0]
print(f"\n📊 annual_trip_volume 총 누적: {grand_total:,}건")

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 셀 6 - (선택) 테이블 초기화
# ============================================================
RESET = False

if RESET:
    with db_open() as con:
        con.execute("DELETE FROM annual_trip_volume")
        cnt = con.execute("SELECT COUNT(*) FROM annual_trip_volume").fetchone()[0]
    print(f"⚠️  초기화 완료: {cnt}건")
else:
    print("RESET=False → 아무 작업 안 함")

In [ ]:
# ============================================================
# 셀 7 - 실패 시군구 재시도
# ============================================================
if not failed:
    print("실패 시군구 없음 — 재시도 불필요")
else:
    print(f"재시도할 실패 시군구: {len(failed)}개")
    retry_failed = []

    for opr_yr, sgg_5, prev_err in failed:
        try:
            items = fetch_all_pages(
                ANNUAL_TRIP_URL,
                AnnualTripVolumeItem,
                extra_params={"opr_yr": opr_yr, "ctpv_cd": "11", "sgg_cd": sgg_5},
            )
            if items:
                df_rows = pd.DataFrame([{
                    "opr_yr": it.opr_yr, "ctpv_cd": it.ctpv_cd,
                    "sgg_cd": it.sgg_cd, "emd_cd": it.emd_cd,
                    "rte_id": it.rte_id, "sttn_seq": it.sttn_seq,
                    "sttn_id": it.sttn_id, "users_type_cd": it.users_type_cd,
                    "tzon": it.tzon, "ride_nope": it.ride_nope, "goff_nope": it.goff_nope,
                } for it in items]).drop_duplicates(
                    subset=["opr_yr","rte_id","sttn_seq","sttn_id","users_type_cd","tzon"]
                )
                with db_open() as con:
                    con.register("df_view", df_rows)
                    con.execute("""
                        INSERT INTO annual_trip_volume
                        SELECT opr_yr, ctpv_cd, sgg_cd, emd_cd,
                               rte_id, sttn_seq, sttn_id,
                               users_type_cd, tzon,
                               ride_nope, goff_nope
                        FROM df_view
                        ON CONFLICT (opr_yr, rte_id, sttn_seq, sttn_id, users_type_cd, tzon) DO NOTHING
                    """)
                    con.unregister("df_view")
                print(f"  ✅ {opr_yr}/{sgg_5}: +{len(df_rows)}건")
            else:
                print(f"  ⚠️  {opr_yr}/{sgg_5}: 빈 응답")
        except Exception as e:
            retry_failed.append((opr_yr, sgg_5, str(e)[:100]))
            print(f"  ❌ {opr_yr}/{sgg_5}: 다시 실패 — {e}")
        time.sleep(0.2)

    failed = retry_failed
    print(f"\n재시도 후 여전히 실패: {len(failed)}개")

In [ ]:
# ============================================================
# 셀 8 - 검증 + 핵심 분석 쿼리
# ============================================================
with db_open(read_only=True) as con:
    # 기본 통계
    print("=== annual_trip_volume 기본 통계 ===")
    print(con.execute("""
        SELECT opr_yr,
               COUNT(*)                AS rows,
               COUNT(DISTINCT rte_id)  AS routes,
               COUNT(DISTINCT sttn_id) AS stops,
               COUNT(DISTINCT sgg_cd)  AS sgg_cnt
        FROM annual_trip_volume
        GROUP BY opr_yr
    """).df())

    # 핵심: 서울 노인(경로) 승차 TOP 10 구
    print("\n=== 서울 노인(경로) 승차 TOP 10 구 ===")
    print(con.execute("""
        SELECT atv.sgg_cd,
               ANY_VALUE(r.locallow_nm) AS sgg_nm,
               SUM(atv.ride_nope)       AS 경로_승차,
               SUM(atv.goff_nope)       AS 경로_하차
        FROM annual_trip_volume atv
        LEFT JOIN region r
          ON r.sido_cd = atv.ctpv_cd
         AND r.sido_cd || r.sgg_cd = atv.sgg_cd
         AND r.umd_cd = '000' AND r.ri_cd = '00'
        WHERE atv.users_type_cd = '04'
        GROUP BY atv.sgg_cd
        ORDER BY 경로_승차 DESC
        LIMIT 10
    """).df())

    # 동작구 노인 시간대별 승차 패턴
    print("\n=== 동작구(11590) 노인 시간대별 승차 (2025 연간) ===")
    print(con.execute("""
        SELECT tzon, SUM(ride_nope) AS 총_승차
        FROM annual_trip_volume
        WHERE sgg_cd='11590' AND users_type_cd='04'
        GROUP BY tzon
        ORDER BY tzon
    """).df())

    # 이용자 유형별 구성 (서울 전체)
    print("\n=== 서울 이용자유형별 승차 구성비 ===")
    print(con.execute("""
        SELECT atv.users_type_cd,
               ANY_VALUE(ut.users_type_nm) AS 유형,
               SUM(atv.ride_nope)          AS 총승차,
               ROUND(100.0 * SUM(atv.ride_nope) / SUM(SUM(atv.ride_nope)) OVER (), 2) AS pct
        FROM annual_trip_volume atv
        LEFT JOIN user_type ut ON atv.users_type_cd = ut.users_type_cd
        GROUP BY atv.users_type_cd
        ORDER BY 총승차 DESC
    """).df())